In [1]:
import matplotlib.pyplot as plt
from pathlib import Path
import openvino as ov
import librosa
import numpy as np
from transformers import AutoTokenizer
from IPython.display import Audio
import torch

/opt/conda/envs/sparktts/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from openvino_export.bicodec import BiCodecTokenizer, BiCodecDetokenizer
from sparktts.models.bicodec import BiCodec
from openvino_export.wav2vec2 import Wav2Vec2Wrapper
from openvino_export.mel_spectrogram import MelSpectrogram
from optimum.intel.openvino import OVModelForCausalLM
from sparktts.utils.file import load_config

In [3]:
bicodec_config = load_config("./pretrained_models/Spark-TTS-0.5B/BiCodec/config.yaml")["audio_tokenizer"]
bicodec_config["mel_params"]

{'sample_rate': 16000, 'n_fft': 1024, 'win_length': 640, 'hop_length': 320, 'mel_fmin': 10, 'mel_fmax': None, 'num_mels': 128}

In [4]:
llm = OVModelForCausalLM.from_pretrained("./pretrained_models/Spark-TTS-0.5B/LLM")

No OpenVINO files were found for ./pretrained_models/Spark-TTS-0.5B/LLM, setting `export=True` to convert the model to the OpenVINO IR. Don't forget to save the resulting model with `.save_pretrained()`
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)
/opt/conda/envs/sparktts/lib/python3.12/site-packages/optimum/exporters/openvino/model_patcher.py:552: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if sequence_length != 1:


In [5]:
llm_tokenizer = AutoTokenizer.from_pretrained("./pretrained_models/Spark-TTS-0.5B/LLM")

In [6]:
llm.save_pretrained("./openvino_models/Spark-TTS-0.5B/LLM")
llm_tokenizer.save_pretrained("./openvino_models/Spark-TTS-0.5B/LLM")

('./openvino_models/Spark-TTS-0.5B/LLM/tokenizer_config.json',
 './openvino_models/Spark-TTS-0.5B/LLM/special_tokens_map.json',
 './openvino_models/Spark-TTS-0.5B/LLM/vocab.json',
 './openvino_models/Spark-TTS-0.5B/LLM/merges.txt',
 './openvino_models/Spark-TTS-0.5B/LLM/added_tokens.json',
 './openvino_models/Spark-TTS-0.5B/LLM/tokenizer.json')

In [7]:
wav2vec = Wav2Vec2Wrapper("./pretrained_models/Spark-TTS-0.5B/wav2vec2-large-xlsr-53")
bicodec = BiCodec.load_from_checkpoint("./pretrained_models/Spark-TTS-0.5B/BiCodec")
mel_spectrogram = MelSpectrogram(bicodec_config["mel_params"])

Missing tensor: mel_transformer.spectrogram.window
Missing tensor: mel_transformer.mel_scale.fb


In [8]:
tokenizer = BiCodecTokenizer(bicodec)
detokenizer = BiCodecDetokenizer(bicodec)

tokenizer.eval()
detokenizer.eval()

BiCodecDetokenizer(
  (quantizer): FactorizedVectorQuantize(
    (in_project): Conv1d(1024, 8, kernel_size=(1,), stride=(1,))
    (out_project): Conv1d(8, 1024, kernel_size=(1,), stride=(1,))
    (codebook): Embedding(8192, 8)
  )
  (speaker_encoder): SpeakerEncoder(
    (speaker_encoder): ECAPA_TDNN(
      (layer1): Conv1dReluBn(
        (conv): Conv1d(128, 512, kernel_size=(5,), stride=(1,), padding=(2,))
        (bn): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (layer2): SE_Res2Block(
        (se_res2block): Sequential(
          (0): Conv1dReluBn(
            (conv): Conv1d(512, 512, kernel_size=(1,), stride=(1,))
            (bn): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          )
          (1): Res2Conv1dReluBn(
            (convs): ModuleList(
              (0-6): 7 x Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
            )
            (bns): ModuleList(
 

In [9]:
audio, sr = librosa.load("./example/prompt_audio.wav", sr=16000)
audio = librosa.util.normalize(audio)
# truncate/pad to 6s (16kHz) audio
audio = np.pad(audio, (0, max(0, 96000 - len(audio))), mode='constant')[:96000]
audio.shape, sr, audio.max(), audio.min(), audio.mean(), audio.std()

((96000,),
 16000,
 np.float32(0.99225426),
 np.float32(-1.0),
 np.float32(-5.6732217e-05),
 np.float32(0.2151192))

In [10]:
feat_input = torch.tensor(audio).unsqueeze(0)
feat = wav2vec(feat_input)
feat.shape, feat.max(), feat.min(), feat_input.shape

(torch.Size([1, 299, 1024]),
 tensor(610.3237, grad_fn=<MaxBackward1>),
 tensor(-237.5022, grad_fn=<MinBackward1>),
 torch.Size([1, 96000]))

In [11]:
mel_input = torch.tensor(audio).unsqueeze(0).unsqueeze(0)
mel = mel_spectrogram(mel_input)
mel.shape, mel.max(), mel.min(), mel_input.shape

(torch.Size([1, 128, 302]),
 tensor(6.0135),
 tensor(2.7275e-06),
 torch.Size([1, 1, 96000]))

In [31]:
semantic_tokens, global_tokens = tokenizer(feat, mel)
semantic_tokens.shape, global_tokens.shape, semantic_tokens.dtype, global_tokens.dtype

(torch.Size([1, 299]), torch.Size([1, 1, 32]), torch.int64, torch.int32)

In [13]:
# # truncate semantic tokens to 50 tokens(1s)
# semantic_tokens = semantic_tokens[:, 50:100]
# semantic_tokens.shape, semantic_tokens.max(), semantic_tokens.min()

In [14]:
wav = detokenizer(semantic_tokens, global_tokens)
wav.shape, wav.max(), wav.min()

(torch.Size([1, 1, 95680]),
 tensor(0.4680, grad_fn=<MaxBackward1>),
 tensor(-0.5621, grad_fn=<MinBackward1>))

In [15]:
Audio(audio, rate=sr)  # Play the audio to verify it loaded correctly

In [16]:
cpu_wav = wav.detach().cpu().numpy().squeeze().squeeze()
Audio(cpu_wav, rate=sr)  # Play the detokenized audio to

In [17]:
ov_mel_spectrogram = ov.convert_model(mel_spectrogram, example_input=mel_input)

/app/openvino_export/mel_spectrogram.py:79: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if wav_with_channel.ndim != 3 or wav_with_channel.shape[1] != 1:


In [18]:
ov_wav2vec = ov.convert_model(wav2vec, example_input=feat_input)

/opt/conda/envs/sparktts/lib/python3.12/site-packages/transformers/models/wav2vec2/modeling_wav2vec2.py:872: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if attn_output.size() != (bsz, self.num_heads, tgt_len, self.head_dim):


In [19]:
ov_tokenizer = ov.convert_model(tokenizer, example_input=(feat, mel))

/app/sparktts/modules/fsq/residual_fsq.py:253: TracerWarning: Iterating over a tensor might cause the trace to be incorrect. Passing a tensor of different shape won't change the number of iterations executed (and might lead to errors or silently give incorrect results).
  zip(self.layers, self.scales)
/app/sparktts/modules/fsq/finite_scalar_quantization.py:201: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  z.shape[-1] == self.dim
/app/sparktts/modules/fsq/finite_scalar_quantization.py:154: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert zhat.shape

In [20]:
ov_detokenizer = ov.convert_model(detokenizer, example_input=(semantic_tokens, global_tokens))

/app/sparktts/modules/fsq/residual_fsq.py:140: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert Q_indices == self.num_quantizers and self.num_quantizers == 1, \
/app/sparktts/modules/blocks/layers.py:65: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if pad > 0:


In [21]:
# Save the OpenVINO models
ov.save_model(ov_mel_spectrogram, "./openvino_models/Spark-TTS-0.5B/mel_spectrogram.xml")
ov.save_model(ov_wav2vec, "./openvino_models/Spark-TTS-0.5B/wav2vec.xml")
ov.save_model(ov_tokenizer, "./openvino_models/Spark-TTS-0.5B/tokenizer.xml")
ov.save_model(ov_detokenizer, "./openvino_models/Spark-TTS-0.5B/detokenizer.xml")

In [22]:
# Compile the OpenVINO model
device_name = "CPU"
core = ov.Core()
ovc_mel_spectrogram = core.compile_model(ov_mel_spectrogram, device_name)
ovc_wav2vec = core.compile_model(ov_wav2vec, device_name)
ovc_tokenizer = core.compile_model(ov_tokenizer, device_name)
ovc_detokenizer = core.compile_model(ov_detokenizer, device_name)

In [23]:
# inference with OpenVINO
ov_audio = librosa.load("./ref.wav", sr=16000)[0]
# make ov_audio is the same shape as audio
# pad or trim
if len(ov_audio) < len(audio):
    ov_audio = np.pad(ov_audio, (0, len(audio) - len(ov_audio)), mode='constant')
elif len(ov_audio) > len(audio):
    ov_audio = ov_audio[:len(audio)]
ov_audio = librosa.util.normalize(ov_audio)
ov_audio.shape, ov_audio.max(), ov_audio.min(), ov_audio.mean(), ov_audio.std()

((96000,),
 np.float32(0.9372959),
 np.float32(-1.0),
 np.float32(-5.2926272e-05),
 np.float32(0.18153115))

In [24]:
ov_mel_input = torch.tensor(ov_audio).unsqueeze(0).unsqueeze(0)
ov_mel = ovc_mel_spectrogram(ov_mel_input) 
ov_mel[0].shape, ov_mel[0].max(), ov_mel[0].min()


((1, 128, 302), np.float32(4.521984), np.float32(2.9008684e-06))

In [25]:
ov_feat_input = torch.tensor(ov_audio).unsqueeze(0)
ov_feat = ovc_wav2vec(ov_feat_input)
ov_feat[0].shape, ov_feat[0].max(), ov_feat[0].min()

((1, 299, 1024), np.float32(595.8177), np.float32(-234.75))

In [26]:
ov_tokens = ovc_tokenizer((ov_feat[0], ov_mel[0]))

In [32]:
ov_semantic_tokens = ov_tokens[0]
ov_global_tokens = ov_tokens[1]
ov_semantic_tokens.shape, ov_global_tokens.shape, ov_semantic_tokens.dtype, ov_global_tokens.dtype

((1, 299), (1, 1, 32), dtype('int64'), dtype('int32'))

In [28]:
ov_wav = ovc_detokenizer((ov_semantic_tokens, ov_global_tokens))
ov_wav[0].shape, ov_wav[0].max(), ov_wav[0].min()

((1, 1, 95680), np.float32(0.4547007), np.float32(-0.44960845))

In [29]:
Audio(ov_audio, rate=sr)  # Play the audio to verify it loaded correctly

In [30]:
ov_cpu_wav = ov_wav[0].squeeze().squeeze()
Audio(ov_cpu_wav, rate=sr)  # Play the detokenized